## Building a GPT

Companion notebook to the Zero to Hero video on GPT

In [8]:
!pip install wget


[notice] A new release of pip available: 22.2.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [1]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

zsh:1: command not found: wget


In [2]:
#read it in to inspect it
with open('input.txt','r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print("length of dataset in characters:", len(text))

length of dataset in characters: 1115394


In [4]:
#get the unique chracters in the dataset
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


### Now we need to create a tokenizer (encode and decoder)

In [5]:
#create a mapping from characters to integers
stoi = { ch:i for i, ch in enumerate(chars)}
itos = { i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] #encoder : takes a string and outputs a integers
decode = lambda l: ''.join(itos[i] for i in l) #decoder : takes a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


### Encoding entire input.txt

In [6]:
!pip install torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 MB 9.1 MB/s eta 0:00:00:00:010:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 17.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 9.3 MB/s eta 0:00:00a 0:00:010m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 9.7 MB/s eta 0:00:00a 0:00:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 16.1 MB/s eta 0:00:00

[notice] A new release of pip available: 22.2.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [7]:
import torch
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.type)
print(data[:100])

/Users/prajwalhiremath/Downloads/practice/nano-gpt/venv/lib/python3.10/site-packages/torch/_subclasses/functional_tensor.py:276: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


torch.Size([1115394]) <built-in method type of Tensor object at 0x1068998f0>
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [14]:
print(data.shape)
#dimension of data
print(data.ndim)

torch.Size([1115394])
1


In [15]:
#now we are going to split the data into train and validation sets
n = int(0.9 * len(data)) #first 90% data will be train, resl validation
train_data = data[:n]
val_data = data[n:]

In [16]:
block_size = 8 #how many characters to predict at once
#we will take a small chunk of data, and try to predict the next character
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [17]:
x = train_data[:block_size] #input
y = train_data[1:block_size+1]#target
for t in range(block_size):
    context = x[:t+1] #input to the model
    target = y[t] #target to predict
    print(f"when input is {context}, target is {target} which is {itos[target.item()]}")    

when input is tensor([18]), target is 47 which is i
when input is tensor([18, 47]), target is 56 which is r
when input is tensor([18, 47, 56]), target is 57 which is s
when input is tensor([18, 47, 56, 57]), target is 58 which is t
when input is tensor([18, 47, 56, 57, 58]), target is 1 which is  
when input is tensor([18, 47, 56, 57, 58,  1]), target is 15 which is C
when input is tensor([18, 47, 56, 57, 58,  1, 15]), target is 47 which is i
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), target is 58 which is t


In [18]:
torch.manual_seed(1337) # for reproducibility
batch_size = 4 #how many independent sequences will we process in parallel
block_size = 8 #what is the maximum context length for predictions
#we will take a small chunk of data, and try to predict the next character

def get_batch(split):
    #generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,)) #randomly pick starting points for each sequence
    x = torch.stack([data[i:i+block_size] for i in ix]) #   input batch of shape (batch_size, block_size)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) #target batch of shape (batch_size, block_size)
    return x, y

xb, yb = get_batch('train')
print("input:", xb.shape, xb.dtype) 
print("target:", yb.shape, yb.dtype)

print('------------')

for b in range(batch_size): # iterate over the batch
    for t in range(block_size): # iterate over the sequence
        context = xb[b, :t+1] #input to the model
        target = yb[b, t] #target to predict
        print(f"when input is {context}, target is {target} which is {itos[target.item()]}")

input: torch.Size([4, 8]) torch.int64
target: torch.Size([4, 8]) torch.int64
------------
when input is tensor([24]), target is 43 which is e
when input is tensor([24, 43]), target is 58 which is t
when input is tensor([24, 43, 58]), target is 5 which is '
when input is tensor([24, 43, 58,  5]), target is 57 which is s
when input is tensor([24, 43, 58,  5, 57]), target is 1 which is  
when input is tensor([24, 43, 58,  5, 57,  1]), target is 46 which is h
when input is tensor([24, 43, 58,  5, 57,  1, 46]), target is 43 which is e
when input is tensor([24, 43, 58,  5, 57,  1, 46, 43]), target is 39 which is a
when input is tensor([44]), target is 53 which is o
when input is tensor([44, 53]), target is 56 which is r
when input is tensor([44, 53, 56]), target is 1 which is  
when input is tensor([44, 53, 56,  1]), target is 58 which is t
when input is tensor([44, 53, 56,  1, 58]), target is 46 which is h
when input is tensor([44, 53, 56,  1, 58, 46]), target is 39 which is a
when input is

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337) # for reproducibility

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)  # embedding table for tokens

    def forward(self, idx, targets): # idx is the input indices, targets are the target indices
        # idx shape: (batch_size, block_size)
        # targets shape: (batch_size, block_size)
        # logits shape: (batch_size, block_size, vocab_size)
        # we will use the token embedding table to get the logits for each token in the vocabulary
        # idx is the input indices, targets are the target indices
        logits = self.token_embedding_table(idx) # (batch_size, block_size, vocab_size)
        B,T,C = logits.shape # B=batch_size, T=block_size, C=vocab_size
        logits = logits.view(B*T, C) 
        targets = targets.view(B*T) #we can also use targets.view(-1) to flatten the targets
        loss = F.cross_entropy(logits, targets)
        return logits, loss

In [27]:
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)  
print(loss)

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)
